In [3]:
#!/usr/bin/env python3
"""
rebuild_pathway_layer.py
=========================
Builds pathway_layer.tsv using Reactome intercellular interactions only.
No KEGG. Optional LR pair inclusion.
"""
from pathlib import Path
from collections import defaultdict
import pandas as pd
import re
import requests

# ── File paths ────────────────────────────────────────────────────────────────
MULTIPLEX_DIR          = Path("../multixrank_input_consolidated/multiplex/gene_multiplex")
REACTOME_INTERACTIONS  = Path("reactome_interactions_human.txt")
UNIPROT_SYM_CSV        = Path("../../og_gene_set/ppi_chloe/d_uniprot_to_symbol_20251120.csv")
UNIPROT_SUBCELL_TSV    = Path("uniprot_subcellular_human.tsv")
LR_PAIRS_CSV           = Path("human_lr_pairs.csv")
OUTPUT_TSV             = MULTIPLEX_DIR / "pathway_layer.tsv"

# ── Toggle ────────────────────────────────────────────────────────────────────
INCLUDE_LR_PAIRS       = True    # set False to exclude LR-derived edges

# ── Step 1 — UniProt → symbol ─────────────────────────────────────────────────
print("=" * 70)
print("STEP 1 — Load UniProt → gene symbol map")
print("=" * 70)
sym_df = pd.read_csv(UNIPROT_SYM_CSV)
sym_df.columns = sym_df.columns.str.strip().str.lower()
uid_col = next(c for c in sym_df.columns if c in ("uniprotid", "uniprot_id", "uniprot"))
sym_col = next(c for c in sym_df.columns if c in ("symbol", "gene_symbol", "genesymbol"))

uniprot_to_symbol: dict[str, str] = {}
for _, row in sym_df.iterrows():
    uid = str(row[uid_col]).strip().upper()
    sym = str(row[sym_col]).strip().upper()
    if uid and sym and uid != "NAN" and sym != "NAN":
        uniprot_to_symbol[uid] = sym
print(f"  Loaded {len(uniprot_to_symbol):,} UniProt → symbol entries")

# ── Step 2 — Multiplex symbols ────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2 — Load multiplex gene symbols")
print("=" * 70)
multiplex_symbols: set[str] = set()
layer_files = sorted(MULTIPLEX_DIR.glob("*_layer.tsv"))
cell_type_layers = [
    lf for lf in layer_files
    if not any(x in lf.name.lower() for x in ("pathway", "ligand", "receptor"))
]
print(f"  Cell-type layer files: {len(cell_type_layers):,}")
for lf in cell_type_layers:
    sub = pd.read_csv(
        lf, sep="\t", header=None,
        names=["g1", "g2"], dtype=str, on_bad_lines="skip"
    )
    for col in ("g1", "g2"):
        multiplex_symbols.update(sub[col].dropna().str.strip().str.upper())
multiplex_symbols.discard("NAN")
print(f"  Unique gene symbols: {len(multiplex_symbols):,}")

uniprot_pattern = re.compile(
    r"^[OPQ][0-9][A-Z0-9]{3}[0-9]$"
    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$"
)
n_look_uniprot = sum(1 for s in list(multiplex_symbols)[:500] if uniprot_pattern.match(s))
print(f"  Sanity: {n_look_uniprot}/500 look like UniProt IDs (should be ~0)")

# ── Step 3 — Subcellular locations ───────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 3 — Load UniProt subcellular locations")
print("=" * 70)

if UNIPROT_SUBCELL_TSV.exists():
    print(f"  Using cached: {UNIPROT_SUBCELL_TSV}")
    subcell_df = pd.read_csv(UNIPROT_SUBCELL_TSV, sep='\t')
    subcell_df.columns = ['uniprot_id', 'subcellular_location']
else:
    print("  Downloading from UniProt REST API...")
    url = (
        'https://rest.uniprot.org/uniprotkb/stream?'
        'query=organism_id:9606+AND+reviewed:true'
        '&fields=accession,cc_subcellular_location'
        '&format=tsv'
    )
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with open(UNIPROT_SUBCELL_TSV, 'w') as f:
        f.write(response.text)
    subcell_df = pd.read_csv(UNIPROT_SUBCELL_TSV, sep='\t')
    subcell_df.columns = ['uniprot_id', 'subcellular_location']

intercellular_terms = [
    'Secreted', 'Extracellular space', 'Extracellular matrix',
    'Cell membrane', 'Plasma membrane'
]
pattern = '|'.join(intercellular_terms)
mask = subcell_df['subcellular_location'].str.contains(pattern, case=False, na=False)
extracellular_proteins = set(subcell_df[mask]['uniprot_id'].str.strip().str.upper())
print(f"  Extracellular/membrane proteins: {len(extracellular_proteins):,}")

# ── Step 4 — Load and filter Reactome interactions ────────────────────────────
print("\n" + "=" * 70)
print("STEP 4 — Load Reactome interactions (intercellular filter)")
print("=" * 70)
reactome_edges: set[tuple[str, str]] = set()

if REACTOME_INTERACTIONS.exists():
    df_r = pd.read_csv(REACTOME_INTERACTIONS, sep="\t")
    df_r.columns = df_r.columns.str.strip()
    print(f"  Raw rows: {len(df_r):,}")

    uniprot_cols = [c for c in df_r.columns if "uniprot" in c.lower()]
    col1, col2   = uniprot_cols[0], uniprot_cols[1]

    uid1 = (
        df_r[col1].astype(str)
        .str.replace("uniprotkb:", "", regex=False)
        .str.strip().str.upper()
    )
    uid2 = (
        df_r[col2].astype(str)
        .str.replace("uniprotkb:", "", regex=False)
        .str.strip().str.upper()
    )

    # Intercellular filter: both proteins must be extracellular/membrane
    mask_inter = uid1.isin(extracellular_proteins) & uid2.isin(extracellular_proteins)
    uid1_f = uid1[mask_inter]
    uid2_f = uid2[mask_inter]
    print(f"  After intercellular filter: {mask_inter.sum():,} rows")

    # Map to gene symbols and filter to multiplex
    sym1 = uid1_f.map(uniprot_to_symbol)
    sym2 = uid2_f.map(uniprot_to_symbol)

    both_mapped = sym1.notna() & sym2.notna()
    both_in_mp  = sym1.isin(multiplex_symbols) & sym2.isin(multiplex_symbols)
    mask_final  = both_mapped & both_in_mp

    print(f"  Both symbols mapped:        {both_mapped.sum():,}")
    print(f"  Both in multiplex:          {both_in_mp.sum():,}")
    print(f"  Both mapped AND in multiplex: {mask_final.sum():,}")

    for s1, s2 in zip(sym1[mask_final], sym2[mask_final]):
        if s1 != s2:
            reactome_edges.add((min(s1, s2), max(s1, s2)))

    print(f"  Unique undirected Reactome edges: {len(reactome_edges):,}")
else:
    print(f"  ⚠ {REACTOME_INTERACTIONS} not found")

# ── Step 5 — Optionally add LR pair edges ────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 5 — LR pairs")
print("=" * 70)
lr_edges: set[tuple[str, str]] = set()

if INCLUDE_LR_PAIRS and LR_PAIRS_CSV.exists():
    lr_df = pd.read_csv(LR_PAIRS_CSV)
    for _, row in lr_df.iterrows():
        lig = str(row.get('ligand_gname',  '')).strip().upper()
        rec = str(row.get('receptor_gname', '')).strip().upper()
        if (lig and rec and lig != 'NAN' and rec != 'NAN'
                and lig in multiplex_symbols and rec in multiplex_symbols
                and lig != rec):
            lr_edges.add((min(lig, rec), max(lig, rec)))
    print(f"  LR pair edges (both in multiplex): {len(lr_edges):,}")
else:
    print("  LR pairs excluded (INCLUDE_LR_PAIRS=False or file not found)")

# ── Step 6 — Merge and report ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 6 — Merge edges")
print("=" * 70)
all_edges         = reactome_edges | lr_edges
reactome_only_set = reactome_edges - lr_edges
lr_only_set       = lr_edges - reactome_edges
both_set          = reactome_edges & lr_edges

print(f"  Reactome edges:    {len(reactome_edges):,}")
print(f"  LR edges:          {len(lr_edges):,}")
print(f"  In both:           {len(both_set):,}")
print(f"  Reactome only:     {len(reactome_only_set):,}")
print(f"  LR only:           {len(lr_only_set):,}")
print(f"  Total unique:      {len(all_edges):,}")

# ── Step 7 — Sanity checks ───────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 7 — Sanity checks")
print("=" * 70)
edge_genes = {g for pair in all_edges for g in pair}
not_in_mp  = edge_genes - multiplex_symbols
print(f"  Genes in edge list:              {len(edge_genes):,}")
print(f"  Genes NOT in multiplex (→ 0):   {len(not_in_mp):,}")
self_pairs = sum(1 for a, b in all_edges if a == b)
print(f"  Self-pairs (→ 0):               {self_pairs:,}")

from collections import Counter
degree = Counter()
for a, b in all_edges:
    degree[a] += 1
    degree[b] += 1
deg_series = pd.Series(degree)
print(f"\n  Degree — min:{deg_series.min()}  "
      f"median:{deg_series.median():.0f}  "
      f"mean:{deg_series.mean():.1f}  "
      f"max:{deg_series.max()}")
print(f"  Top 10 hubs:")
for gene, deg in deg_series.sort_values(ascending=False).head(10).items():
    print(f"    {gene:15s}: {deg:,}")

no_edge = multiplex_symbols - edge_genes
print(f"\n  Multiplex genes with no edge: {len(no_edge):,} "
      f"({100*len(no_edge)//max(len(multiplex_symbols),1)}%)")

# ── Step 8 — Write ───────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 8 — Write pathway_layer.tsv")
print("=" * 70)
sorted_edges = sorted(all_edges)
with open(OUTPUT_TSV, "w") as f:
    for a, b in sorted_edges:
        f.write(f"{a}\t{b}\n")
print(f"  Written {len(sorted_edges):,} edges → {OUTPUT_TSV}")

verify = pd.read_csv(OUTPUT_TSV, sep="\t", header=None, names=["g1", "g2"], dtype=str)
print(f"  Verification read-back: {len(verify):,} rows")

STEP 1 — Load UniProt → gene symbol map
  Loaded 165,580 UniProt → symbol entries

STEP 2 — Load multiplex gene symbols
  Cell-type layer files: 64
  Unique gene symbols: 17,051
  Sanity: 1/500 look like UniProt IDs (should be ~0)

STEP 3 — Load UniProt subcellular locations
  Using cached: uniprot_subcellular_human.tsv
  Extracellular/membrane proteins: 5,766

STEP 4 — Load Reactome interactions (intercellular filter)
  Raw rows: 123,895
  After intercellular filter: 19,788 rows
  Both symbols mapped:        19,562
  Both in multiplex:          18,041
  Both mapped AND in multiplex: 18,041
  Unique undirected Reactome edges: 4,336

STEP 5 — LR pairs
  LR pair edges (both in multiplex): 2,267

STEP 6 — Merge edges
  Reactome edges:    4,336
  LR edges:          2,267
  In both:           475
  Reactome only:     3,861
  LR only:           1,792
  Total unique:      6,128

STEP 7 — Sanity checks
  Genes in edge list:              2,220
  Genes NOT in multiplex (→ 0):   0
  Self-pairs (→

In [4]:
#!/usr/bin/env python3
"""
verify_pathway_layer.py
========================
Sanity checks on pathway_layer.tsv — Reactome intercellular only, no KEGG.
"""
from pathlib import Path
from collections import Counter
import pandas as pd
import networkx as nx
import re

# ── Paths ─────────────────────────────────────────────────────────────────────
PATHWAY_LAYER = Path("../multixrank_input_consolidated/multiplex/gene_multiplex/pathway_layer.tsv")
MULTIPLEX_DIR = Path("../multixrank_input_consolidated/multiplex/gene_multiplex")
REACTOME_CSV  = Path("reactome_interactions_with_pathways_v2.csv")

uniprot_pattern = re.compile(
    r"^[OPQ][0-9][A-Z0-9]{3}[0-9]$"
    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}$"
)

# ── 1. Load pathway layer ─────────────────────────────────────────────────────
print("=" * 70)
print("1. LOAD PATHWAY LAYER")
print("=" * 70)
df = pd.read_csv(PATHWAY_LAYER, sep="\t", header=None, names=["g1", "g2"], dtype=str)
print(f"  Rows:        {len(df):,}")
print(f"  Sample edges:")
print(df.head(10).to_string(index=False))

# ── 2. Basic integrity ────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("2. BASIC INTEGRITY")
print("=" * 70)
self_pairs = (df["g1"] == df["g2"]).sum()
nulls      = df.isna().sum().sum()
df["key"]  = df.apply(lambda r: tuple(sorted([r["g1"], r["g2"]])), axis=1)
dupes      = df.duplicated(subset="key").sum()
all_genes  = set(df["g1"]) | set(df["g2"])
n_look_uid = sum(1 for g in list(all_genes)[:500] if uniprot_pattern.match(g))

print(f"  Self-pairs (→ 0):                  {self_pairs:,}")
print(f"  Null values (→ 0):                 {nulls:,}")
print(f"  Duplicate edges (→ 0):             {dupes:,}")
print(f"  Genes looking like UniProt (→ ~0): {n_look_uid}/500 sampled")
print(f"  Sample genes: {sorted(all_genes)[:15]}")

# ── 3. Coverage vs multiplex ──────────────────────────────────────────────────
print("\n" + "=" * 70)
print("3. COVERAGE VS MULTIPLEX CELL-TYPE LAYERS")
print("=" * 70)
multiplex_symbols: set[str] = set()
layer_files = sorted(MULTIPLEX_DIR.glob("*_layer.tsv"))
cell_type_layers = [
    lf for lf in layer_files
    if not any(x in lf.name.lower() for x in ("pathway", "ligand", "receptor"))
]
print(f"  Cell-type layer files: {len(cell_type_layers):,}")
for lf in cell_type_layers:
    sub = pd.read_csv(lf, sep="\t", header=None,
                      names=["g1", "g2"], dtype=str, on_bad_lines="skip")
    for col in ("g1", "g2"):
        multiplex_symbols.update(sub[col].dropna().str.strip().str.upper())
multiplex_symbols.discard("NAN")

pathway_genes   = set(df["g1"]) | set(df["g2"])
not_in_mp       = pathway_genes - multiplex_symbols
in_mp           = pathway_genes & multiplex_symbols
mp_with_edge    = multiplex_symbols & pathway_genes
mp_without_edge = multiplex_symbols - pathway_genes

print(f"  Total multiplex genes:               {len(multiplex_symbols):,}")
print(f"  Pathway layer genes:                 {len(pathway_genes):,}")
print(f"  In multiplex:                        {len(in_mp):,}")
print(f"  NOT in multiplex (→ 0):             {len(not_in_mp):,}")
print(f"  Multiplex genes WITH edge:           {len(mp_with_edge):,} "
      f"({100*len(mp_with_edge)//max(len(multiplex_symbols),1)}%)")
print(f"  Multiplex genes WITHOUT edge:        {len(mp_without_edge):,} "
      f"({100*len(mp_without_edge)//max(len(multiplex_symbols),1)}%)")
if not_in_mp:
    print(f"  Sample not-in-multiplex: {sorted(not_in_mp)[:15]}")

# ── 4. Graph properties ───────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("4. GRAPH PROPERTIES")
print("=" * 70)
G = nx.from_pandas_edgelist(df, "g1", "g2")
degrees = pd.Series(dict(G.degree()))
lcc     = max(nx.connected_components(G), key=len)

print(f"  Nodes:               {G.number_of_nodes():,}")
print(f"  Edges:               {G.number_of_edges():,}")
print(f"  Connected components:{nx.number_connected_components(G):,}")
print(f"  Largest component:   {len(lcc):,} nodes "
      f"({100*len(lcc)//G.number_of_nodes()}%)")
print(f"\n  Degree — min:{degrees.min()}  "
      f"median:{degrees.median():.0f}  "
      f"mean:{degrees.mean():.1f}  "
      f"max:{degrees.max()}")
print(f"\n  Top 15 hubs:")
for gene, deg in degrees.sort_values(ascending=False).head(15).items():
    print(f"    {gene:15s}: {deg:,}")

# ── 5. Cross-check vs reactome CSV ────────────────────────────────────────────
print("\n" + "=" * 70)
print("5. CROSS-CHECK VS REACTOME CSV")
print("=" * 70)
if REACTOME_CSV.exists():
    rdf = pd.read_csv(REACTOME_CSV)
    print(f"  Reactome CSV rows:        {len(rdf):,}")
    print(f"  Self-pairs in CSV:        {(rdf['symbol1']==rdf['symbol2']).sum():,}")
    print(f"  With common_pathway:      {rdf['common_pathway'].notna().sum():,}")
    if 'is_lr_pair' in rdf.columns:
        print(f"  Flagged as LR pairs:      {rdf['is_lr_pair'].sum():,}")
    if 'n_reaction_contexts' in rdf.columns:
        print(f"  Avg reaction contexts:    {rdf['n_reaction_contexts'].mean():.1f}")

    csv_pairs = set(
        tuple(sorted([r["symbol1"], r["symbol2"]]))
        for _, r in rdf[
            rdf["symbol1"].notna() &
            rdf["symbol2"].notna() &
            (rdf["symbol1"] != rdf["symbol2"])
        ].iterrows()
    )
    layer_pairs      = set(df["key"].tolist())
    in_both          = layer_pairs & csv_pairs
    in_layer_not_csv = layer_pairs - csv_pairs
    in_csv_not_layer = csv_pairs - layer_pairs

    print(f"\n  Pairs in pathway_layer.tsv:      {len(layer_pairs):,}")
    print(f"  Distinct pairs in reactome CSV:  {len(csv_pairs):,}")
    print(f"  In both:                         {len(in_both):,}")
    print(f"  In layer but NOT in CSV:         {len(in_layer_not_csv):,}")
    print(f"  In CSV but NOT in layer:         {len(in_csv_not_layer):,}")
    print(f"  (^ filtered out: not in multiplex)")

    # Pathway coverage
    rdf_pw = rdf[
        (rdf["symbol1"] != rdf["symbol2"]) & rdf["common_pathway"].notna()
    ].copy()
    rdf_pw["key"] = rdf_pw.apply(
        lambda r: tuple(sorted([r["symbol1"], r["symbol2"]])), axis=1
    )
    pw_covered = layer_pairs & set(rdf_pw["key"])
    print(f"\n  Layer edges with named pathway:  {len(pw_covered):,} "
          f"({100*len(pw_covered)//max(len(layer_pairs),1)}%)")

    print(f"\n  Top 20 pathways in reactome CSV:")
    counter = Counter(
        p for val in rdf_pw["common_pathway"].dropna()
        for p in str(val).split("|") if p.strip()
    )
    for pw, n in counter.most_common(20):
        print(f"    {n:>6,}  {pw}")
else:
    print(f"  ⚠ {REACTOME_CSV} not found")

# ── 6. Summary ────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("6. SUMMARY")
print("=" * 70)
checks = {
    "Self-pairs = 0":        self_pairs == 0,
    "Duplicates = 0":        dupes == 0,
    "No genes outside multiplex": len(not_in_mp) == 0,
    "Null values = 0":       nulls == 0,
}
for check, passed in checks.items():
    print(f"  {'✅' if passed else '❌'} {check}")

all_passed = all(checks.values())
print(f"\n  {'✅ All checks passed' if all_passed else '❌ Issues found — see above'}")
print(f"\n  Total edges: {len(df):,}")
print(f"  Gene coverage: {len(mp_with_edge):,} / {len(multiplex_symbols):,} multiplex genes")

1. LOAD PATHWAY LAYER
  Rows:        6,128
  Sample edges:
   g1     g2
  A2M  KLKB1
  A2M   LRP1
  A2M   MMP1
  A2M  MMP13
  A2M   MMP3
 AAMP   EGFR
 AAMP   NOD2
 AAMP TBXA2R
AANAT MTNR1A
AANAT MTNR1B

2. BASIC INTEGRITY
  Self-pairs (→ 0):                  0
  Null values (→ 0):                 0
  Duplicate edges (→ 0):             0
  Genes looking like UniProt (→ ~0): 1/500 sampled
  Sample genes: ['A2M', 'AAMP', 'AANAT', 'ABCA1', 'ABCA7', 'ABCG5', 'ABCG8', 'ABHD17A', 'ABHD17B', 'ACAN', 'ACE', 'ACE2', 'ACKR1', 'ACKR2', 'ACKR3']

3. COVERAGE VS MULTIPLEX CELL-TYPE LAYERS
  Cell-type layer files: 64
  Total multiplex genes:               17,051
  Pathway layer genes:                 2,220
  In multiplex:                        2,220
  NOT in multiplex (→ 0):             0
  Multiplex genes WITH edge:           2,220 (13%)
  Multiplex genes WITHOUT edge:        14,831 (86%)

4. GRAPH PROPERTIES
  Nodes:               2,220
  Edges:               6,128
  Connected components:105
  Lar